# PBIT Safety — Severity Multiplier Grid Search

**Objective**: quantify how different severity multiplier scales affect CMR and Macro F1.

**Fixed model**: LightGBM, F1-tuned params (`best_lgbm_params.json`), f05_boost=6.98×, no sev3_boost.

**Config**: Aug + custom frequency × severity weighting.

**Scales tested** (multipliers for Sev0 / Sev1 / Sev2 / Sev3):

| Scale | Sev0 | Sev1 | Sev2 | Sev3 | Ratio 3:0 |
|-------|-----:|-----:|-----:|-----:|----------:|
| Freq only (no sev) | 1 | 1 | 1 | 1 | 1× |
| Current (1+sev) | 1 | 2 | 3 | 4 | 4× |
| Intermediate A | 1 | 2 | 4 | 6 | 6× |
| Paper (geometric) | 1 | 2 | 4 | 8 | 8× |
| Intermediate B | 1 | 3 | 6 | 9 | 9× |
| Aggressive | 1 | 2 | 4 | 12 | 12× |
| Extreme | 1 | 2 | 4 | 16 | 16× |

Primary metrics: CMR, Downward Risk, Severity Error Rate.  
Secondary: Macro F1, Balanced Accuracy, F06 Recall, Sev3 Recall.

Results saved to `data/safety_sev_multipliers.csv`.

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks_safety':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb

from pbit.modeling import (
    META_COLS, FAULT_ORDER, SEVERITY_MAP,
    make_cv_folds, summarize_cv,
    evaluate_predictions, compute_safety_metrics,
)
from pbit.data_validation import load_long_dataset
from pbit.augmentation import augment_training_set
from pbit.features import extract_features

TRAIN_PATH      = PROJECT_ROOT / 'data' / 'train_real.csv'
LONG_PATH       = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
F1_PARAMS_PATH  = PROJECT_ROOT / 'data' / 'best_lgbm_params.json'
RESULTS_PATH    = PROJECT_ROOT / 'data' / 'safety_sev_multipliers.csv'

PROJECT_ROOT

## Load Data, Folds & F1-tuned Params

In [ ]:
train_df  = pd.read_csv(TRAIN_PATH)
long_df   = load_long_dataset(LONG_PATH)
feat_cols = [c for c in train_df.columns if c not in META_COLS]
folds     = make_cv_folds(train_df, n_splits=5, random_state=42)

with open(F1_PARAMS_PATH) as f:
    f1_all = json.load(f)
f05_boost = f1_all.pop('f05_boost')
lgb_params = f1_all

print(f'Training set : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'Folds        : {len(folds)}')
print(f'f05_boost    : {f05_boost:.4f}')
print(f'LightGBM params: {lgb_params}')

## Severity Scales & Weight Helper

Each scale maps a severity index (0, 1, 2, 3) to a multiplier.  
Sample weight = frequency_weight × severity_multiplier, with f05_boost applied on top.

In [ ]:
# Scale name → {sev_index: multiplier}
SEV_SCALES = {
    'Freq only':         {0: 1, 1: 1, 2: 1,  3: 1},
    'Current (1+sev)':   {0: 1, 1: 2, 2: 3,  3: 4},
    'Intermediate A':    {0: 1, 1: 2, 2: 4,  3: 6},
    'Paper (geometric)': {0: 1, 1: 2, 2: 4,  3: 8},
    'Intermediate B':    {0: 1, 1: 3, 2: 6,  3: 9},
    'Aggressive':        {0: 1, 1: 2, 2: 4,  3: 12},
    'Extreme':           {0: 1, 1: 2, 2: 4,  3: 16},
}


def make_weights(y_train: np.ndarray, scale: dict, f05_boost: float = 1.0) -> np.ndarray:
    """Compute freq × severity weights with custom scale and f05 boost."""
    classes, counts = np.unique(y_train, return_counts=True)
    n = len(y_train)
    k = len(classes)
    freq_w = {c: n / (k * cnt) for c, cnt in zip(classes, counts)}

    weights = np.empty(n, dtype=float)
    for i, label in enumerate(y_train):
        sev_idx = SEVERITY_MAP.get(label, 0)
        fw = freq_w.get(label, 1.0)
        sw = scale[sev_idx]
        weights[i] = fw * sw
    weights[y_train == 'F05'] *= f05_boost
    return weights


print('Scales defined:', list(SEV_SCALES.keys()))

## Grid Search — 5-fold CV per Scale

In [ ]:
results = []

t0_total = time.time()
for scale_name, scale in SEV_SCALES.items():
    fold_results = []
    fold_cmrs, fold_down, fold_severr = [], [], []

    t0 = time.time()
    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_real  = train_df.iloc[tr_idx]
        val_real = train_df.iloc[val_idx]

        tr_exp_ids = set(tr_real['Experiment_Index'].tolist())
        aug_long   = augment_training_set(
            long_df, tr_exp_ids,
            target_per_class=160, noise_scale=0.01,
            random_state=42 + fold_i,
        )
        if len(aug_long) > 0:
            aug_feat = extract_features(aug_long)
            tr_fold  = pd.concat([tr_real, aug_feat], ignore_index=True)
        else:
            tr_fold = tr_real

        X_train = tr_fold[feat_cols].values.astype(float)
        y_train = tr_fold['Fault_Mode'].values
        X_val   = val_real[feat_cols].values.astype(float)
        y_val   = val_real['Fault_Mode'].values

        sw = make_weights(y_train, scale, f05_boost)

        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_train, y_train, sample_weight=sw)
        y_pred = model.predict(X_val)

        m  = evaluate_predictions(y_val, y_pred)
        sm = compute_safety_metrics(y_val, y_pred)
        m['fold'] = fold_i
        fold_results.append(m)
        fold_cmrs.append(sm['cmr'])
        fold_down.append(sm['down_risk'])
        fold_severr.append(sm['sev_err'])

    s = summarize_cv(fold_results)
    cmr_mean  = float(np.nanmean([v for v in fold_cmrs if not np.isnan(v)] or [float('nan')]))
    down_mean = float(np.mean(fold_down))
    sev_mean  = float(np.mean(fold_severr))

    row = {
        'Scale':       scale_name,
        'Sev3_mult':   scale[3],
        'CMR':         round(cmr_mean, 4),
        'Down_Risk':   round(down_mean, 4),
        'Sev_Err':     round(sev_mean, 4),
        'Macro_F1':    round(s['macro_f1_mean'], 4),
        'Bal_Acc':     round(s['balanced_accuracy_mean'], 4),
        'F06_Recall':  round(s['f06_recall_mean'], 4),
        'F05_Recall':  round(s['f05_recall_mean'], 4),
        'F04_Recall':  round(s['f04_recall_mean'], 4),
        'Sev3_Recall': round(s['sev3_recall_mean'], 4),
    }
    results.append(row)
    elapsed = time.time() - t0
    print(f'{scale_name:<22}  CMR={cmr_mean:.4f}  DR={down_mean:.4f}  SevErr={sev_mean:.4f}  '
          f'F1={s["macro_f1_mean"]:.4f}  F06={s["f06_recall_mean"]:.4f}  ({elapsed:.0f}s)')

print(f'\nTotal time: {(time.time()-t0_total)/60:.1f} min')

## Results Table — sorted by CMR

In [ ]:
tbl = pd.DataFrame(results).sort_values('CMR').reset_index(drop=True)
tbl

## Plots — Safety vs Quality Trade-off

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sort by Sev3_mult for consistent x-axis
plot_df = pd.DataFrame(results).sort_values('Sev3_mult')
x = np.arange(len(plot_df))
labels = [f"{r['Scale']}\n(×{r['Sev3_mult']})" for _, r in plot_df.iterrows()]

ax = axes[0]
bars = ax.bar(x, plot_df['CMR'], color='tomato', alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
ax.set_ylabel('CMR'); ax.set_title('CMR by severity scale (lower = safer)')
ax.grid(True, axis='y', linewidth=0.3)
for bar, v in zip(bars, plot_df['CMR']):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.4f}',
            ha='center', va='bottom', fontsize=7)

ax = axes[1]
ax.bar(x, plot_df['Macro_F1'], color='steelblue', alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
ax.set_ylabel('Macro F1'); ax.set_title('Macro F1 by severity scale')
ax.grid(True, axis='y', linewidth=0.3)
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.text(i, row['Macro_F1'] + 0.001, f'{row["Macro_F1"]:.4f}',
            ha='center', va='bottom', fontsize=7)

ax = axes[2]
sc = ax.scatter(
    plot_df['Macro_F1'], plot_df['CMR'],
    c=plot_df['Sev3_mult'], cmap='RdYlGn_r',
    s=80, zorder=3, edgecolors='black', linewidths=0.5
)
plt.colorbar(sc, ax=ax, label='Sev3 multiplier')
for _, row in plot_df.iterrows():
    ax.annotate(row['Scale'], (row['Macro_F1'], row['CMR']),
                textcoords='offset points', xytext=(4, 4), fontsize=7)
ax.set_xlabel('Macro F1'); ax.set_ylabel('CMR')
ax.set_title('CMR vs Macro F1 (colour = Sev3 multiplier)')
ax.grid(True, linewidth=0.3)

fig.suptitle('Severity multiplier grid search — LightGBM F1-tuned params, Aug + custom Freq×Sev', fontsize=11)
plt.tight_layout()
plt.show()

## Downward Risk & Severity Error by Scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
w = 0.35

ax = axes[0]
ax.bar(x - w/2, plot_df['CMR'],       w, label='CMR',           color='tomato',    alpha=0.85)
ax.bar(x + w/2, plot_df['Down_Risk'], w, label='Downward Risk', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
ax.set_title('CMR and Downward Risk')
ax.legend(fontsize=8); ax.grid(True, axis='y', linewidth=0.3)

ax = axes[1]
ax.bar(x, plot_df['Sev_Err'], color='mediumpurple', alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
ax.set_title('Severity Error Rate')
ax.grid(True, axis='y', linewidth=0.3)

plt.tight_layout()
plt.show()

## Key Scale Comparison — Current vs Paper vs Best CMR

In [ ]:
highlight = ['Freq only', 'Current (1+sev)', 'Paper (geometric)']
best_cmr_scale = tbl.iloc[0]['Scale']
if best_cmr_scale not in highlight:
    highlight.append(best_cmr_scale)

ref_df = pd.DataFrame(results)
ref_df = ref_df[ref_df['Scale'].isin(highlight)].set_index('Scale').loc[highlight]

print('=== Key scale comparison ===')
print(ref_df[['Sev3_mult', 'CMR', 'Down_Risk', 'Sev_Err', 'Macro_F1', 'Bal_Acc', 'F06_Recall', 'Sev3_Recall']].to_string())

if 'Current (1+sev)' in ref_df.index and 'Paper (geometric)' in ref_df.index:
    paper = ref_df.loc['Paper (geometric)']
    curr  = ref_df.loc['Current (1+sev)']
    print(f"\nPaper (geometric) vs Current (1+sev):")
    print(f"  CMR:       {paper['CMR']:.4f} vs {curr['CMR']:.4f}  ({paper['CMR']-curr['CMR']:+.4f})")
    print(f"  Macro F1:  {paper['Macro_F1']:.4f} vs {curr['Macro_F1']:.4f}  ({paper['Macro_F1']-curr['Macro_F1']:+.4f})")
    print(f"  Down Risk: {paper['Down_Risk']:.4f} vs {curr['Down_Risk']:.4f}  ({paper['Down_Risk']-curr['Down_Risk']:+.4f})")

## Save Results

In [ ]:
tbl.to_csv(RESULTS_PATH, index=False)
print('Saved:', RESULTS_PATH)
print()
print('Full results table (sorted by CMR):')
print(tbl[['Scale', 'Sev3_mult', 'CMR', 'Down_Risk', 'Sev_Err', 'Macro_F1', 'Bal_Acc', 'F06_Recall', 'Sev3_Recall']].to_string(index=False))